# Model Quantization

In this chapter we will explore **model quanization** - mapping the weights of a model to lower precision numbers after training. 

## How Quantization Works

_Do we actually need to store the weights of a model in full precision to represent a model's knowledge?_

If we could store the weights not in 32-bit floating point numbers, but in 8-bit integers, we could achieve
- 4x smaller model files
- 2-4x faster inference (multiplication with integers is faster than with floating point numbers)

**How to quantize Float32 into Int8?**

We can map a distribution of 32-bit floating point numbers onto the range of 8-bit integers (256 values) by fitting two parameters:
- a scale factor $S$
- a zero-shift scalar $z$

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/quantization-schema.png" style="max-width: 700px; max-height: 500px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Quantization mapping from Float32 to Int8
  </div>
</div>

**Post-training quantization of weights**

For each layer of the neural network
1. analyze weight distribution and calculate S and z
1. apply quantization formula and store quantized weights

**→ 4 x smaller weights**

_But what happens during computation? So far the activations (the outputs of "neurons") remain in full precision._

**Post-training quantization of activations**

For each layer of the neural network
1. run forward pass with a few samples
1. analyze activation distribution and calculate S and z
 
**→ 2-4 x faster inference**


_Can we go even smaller with quantization?_

Yes, 6-bit, 4-bit or even 2-bit quantization are common. 

But if we reduce the precision of weights so radically, how much of the model's capabilities are we sacrificing?

- This is hard to predict, models vary in their sensitivity to quantization.
- Therefore, the capability loss needs to be evaluated experimentally
- Fortunately, others have typically done some evaluation for us - see the model card for recommended variants

**→ up to 16x smaller model files, but potentially significant quality loss**


## Why Quantization Matters

_The open LLM ecosystem thrives on quantization_
- quantization enables
    - medium-sized models on modest hardware (e.g. 15B parameters in 
9 GB of RAM)
    - easier online distribution
- many models in Ollama catalog are quantized by default!


## A LLM in Quantized Versions

Let's choose a capable LLM for an experiment with quantization:


> A LLama2-7b model trained on Orca Style datasets.
> 
-- [🤗 model card](https://huggingface.co/pankajmathur/orca_mini_v3_7b)

This means that the model is the result of...
-  taking the open [Llama 2](https://llama.meta.com/llama2/) base model...
- and improving its reasoning and instruction following performance through [fine-tuning on step-by-step explanations generated by a large teacher model](https://arxiv.org/abs/2306.02707)

We are now going to get some quantized versions of this model. A 🤗 user has already applied various quantization methods to the model and provided the results.

In [ ]:
from ai_dojo import show

In [ ]:
import huggingface_hub

In [ ]:
huggingface_user = "TheBloke"
model_name = "orca_mini_v3_7B-GGUF"
full_model_name = f"{huggingface_user}/{model_name}"
model_page_url= f"https://huggingface.co/{full_model_name}"
model_page_url

The model page lists a variety of quantized models in different quantization types. Let's focus on a few.

In [ ]:
quantization_types = [
    "Q8_0",
    "Q4_K_M",
    "Q2_K",
]

These quantized models are not available from the Ollama model registry. Fortunately, adding custom models downloaded from 🤗 to Ollama is quite straightforward. We need to give Ollama a model name and a simple configuration file.

In [ ]:
import tqdm

In [ ]:
model_file_base_name = "orca_mini_v3_7b"


In [ ]:
quantized_model_paths = {}
for qtype in tqdm.tqdm(quantization_types):
    model_file = f"{model_file_base_name}.{qtype}.gguf" 
    tqdm.tqdm.write(f"Downloading {qtype} model")
    model_path = huggingface_hub.hf_hub_download(full_model_name, filename=model_file)
    quantized_model_paths[qtype] = model_path
    
quantized_model_paths

In [ ]:
import ollama

In [ ]:
def make_model_file(model_path) -> str:
    """Creates a simple Ollama model configuration file for the given serialized model."""
    content = f"FROM {model_path}"
    return content

In [ ]:
for qtype, model_path in quantized_model_paths.items():
    ollama_model_name = f"{model_name}:{qtype}"
    print(f"Creating Ollama model {ollama_model_name}")
    response = ollama.create(
        model=ollama_model_name,
        modelfile=make_model_file(model_path)
    )
    print(response["status"])


Let's verify that all of the models above are now available in Ollama. Also, have a look at the size on disk.

In [ ]:
import pandas
from pandas import DataFrame


def show_ollama_model_table() -> DataFrame:
    """List Ollama models in tabular form"""
    model_data = pandas.json_normalize(ollama.list()["models"])
    model_data = model_data[["model", "size", "details.family", "details.format", "details.parameter_size", "details.quantization_level"]]
    model_data["size"] = model_data["size"].apply(lambda s: round(s / (1024**3), 2))
    return model_data

In [ ]:
show_ollama_model_table()

## Test

Let's prompt the original and the quantized models to do something useful. This should give us a first impression of how model performance relates to the level of quantization.

In [ ]:
test_prompt = "Create an agenda for a two-day workshop on Generative AI"
show.text(test_prompt)

### 8-bit 

In [ ]:
quantization_level = "Q8_0"
ollama_model_name = f"{model_name}:{quantization_level}"
ollama_model_name

In [ ]:
response = ollama.chat(
    model=ollama_model_name,
    messages=[
        {
            "role": "user",
            "content": test_prompt,
        },
    ],
    stream=True,
)
show.stream(response)

### 4-bit

In [ ]:
quantization_level = "Q4_K_M"
ollama_model_name = f"{model_name}:{quantization_level}"
ollama_model_name

In [ ]:
response = ollama.chat(
    model=ollama_model_name,
    messages=[
        {
            "role": "user",
            "content": test_prompt,
        },
    ],
    stream=True,
)
show.stream(response)

### 2-bit

In [ ]:
quantization_level = "Q2_K"
ollama_model_name = f"{model_name}:{quantization_level}"
ollama_model_name

In [ ]:
response = ollama.chat(
    model=ollama_model_name,
    messages=[
        {
            "role": "user",
            "content": test_prompt,
        },
    ],
    stream=True,
)
show.stream(response)

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2024-2026 [Christian Staudt](https://clstaudt.me) and [Katharina Rasch](https://krasch.io)_